# Logistic Regression from Scratch
Binary classification with NumPy and gradient descent.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

In [ ]:
X = np.array([[1,1],[1.5,1.2],[2,1.8],[2.2,2],[2.5,2.3],[3,2.8],
              [3.2,3],[3.5,3.2],[4,3.8],[4.5,4],[5,4.5],[5.5,5]])
y = np.array([0]*6 + [1]*6)

for c in (0, 1):
    plt.scatter(*X[y == c].T, s=70, label=f"Class {c}")
plt.xlabel("Feature 1"); plt.ylabel("Feature 2"); plt.title("Dataset"); plt.legend(); plt.show()

### Model
$p = \sigma(Xw + b)$, cost = binary cross-entropy, gradients $dw = \frac{1}{m}X^T(p - y)$ and $db = \frac{1}{m}\sum(p - y)$.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def predict_proba(X, w, b):
    return sigmoid(X @ w + b)

def compute_cost(X, y, w, b):
    p = np.clip(predict_proba(X, w, b), 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def compute_gradients(X, y, w, b):
    err = predict_proba(X, w, b) - y
    return X.T @ err / len(y), err.mean()

def train(X, y, lr=0.1, iters=1000):
    w, b, history = np.zeros(X.shape[1]), 0.0, []
    for _ in range(iters):
        history.append(compute_cost(X, y, w, b))
        dw, db = compute_gradients(X, y, w, b)
        w, b = w - lr * dw, b - lr * db
    return w, b, history

def predict(X, w, b, threshold=0.5):
    return (predict_proba(X, w, b) >= threshold).astype(int)

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

In [ ]:
w, b, history = train(X, y)
print("Initial cost:", round(history[0], 4), "| Final cost:", round(history[-1], 4))
print("Weights:", w, "| Bias:", b)
print("Predictions:", predict(X, w, b))
print("Accuracy:", accuracy(y, predict(X, w, b)))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(history); ax[0].set(xlabel="Iteration", ylabel="Cost", title="Cost during training")

for c in (0, 1):
    ax[1].scatter(*X[y == c].T, s=70, label=f"Class {c}")
x1 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 100)
ax[1].plot(x1, -(w[0] * x1 + b) / w[1], "r", label="Decision boundary")
ax[1].set(xlabel="Feature 1", ylabel="Feature 2", title="Decision boundary"); ax[1].legend()
plt.tight_layout(); plt.show()

### Experiments: learning rate and decision threshold

In [ ]:
for lr in (0.001, 0.1, 1.0):
    w_, b_, h = train(X, y, lr)
    print(f"lr={lr:<6} final cost={h[-1]:.4f}  accuracy={accuracy(y, predict(X, w_, b_)):.2f}")

for t in (0.3, 0.5, 0.7):
    print(f"threshold={t}  accuracy={accuracy(y, predict(X, w, b, t)):.2f}")